# Chapter 1: From text to tokens

Computers do not understand words, they only understand numbers. So before we can train any kind of language model, we first have to turn our text into numbers that it can work with.

That is what this notebook is all about. We start with plain English text and slowly build our way up to the numbers a model can learn from. Here is the path we will follow:

- read a book into Python as raw text
- break that text into small pieces called tokens
- give every unique token its own ID
- write a small tokenizer that goes from text to IDs and back
- switch to a smarter, industry standard tokenizer called BPE
- finally turn those IDs into input and target pairs and embeddings, ready for training

The cell below grabs the text data we will use, a short story called The Verdict.

In [4]:
import os
import urllib.request

file_path = "../data/the-verdict.txt"
if not os.path.exists(file_path):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    urllib.request.urlretrieve(url, file_path)

## Step one: read the text

Let us open the file we just downloaded and load it into a variable. We also print the total number of characters and the first hundred of them, just to confirm it loaded correctly and to get a feel for what the text looks like.

In [5]:
with open("../data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text=f.read()
print("total character count:", len(raw_text))
print(raw_text[:99])

total character count: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


## Breaking text into tokens

A token is just a small piece of text. It might be a whole word, a punctuation mark, or sometimes only a part of a word. Before splitting the whole story, let us first understand how splitting works with a tiny example.

Here we split a short sentence on its spaces. Notice that the spaces are kept in the result instead of being thrown away. That matters, because later on we care about punctuation and spacing too, not just the words.

In [6]:
import re
text="hello, world. this, is a test"
result= re.split(r'(\s)', text)
print(result)

['hello,', ' ', 'world.', ' ', 'this,', ' ', 'is', ' ', 'a', ' ', 'test']


## Tokenizing the whole story

Now we use the same splitting idea on the full text, but with a slightly better rule. This time we split on spaces as well as punctuation such as commas, periods and question marks. We also drop the empty pieces and the leftover spaces.

What we are left with is a clean list of tokens for the entire book.

In [7]:
preprocessed = re.split(r'([,.?_!"()\']|--|\s)', raw_text)

preprocessed = [item.strip() for item in preprocessed if item.strip()]
print("total token in the first book text", len(preprocessed))
print("first 20 characters", preprocessed[:20])

total token in the first book text 4649
first 20 characters ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was']


## Building a vocabulary

A model cannot work with the word "hello" directly, it needs a number for it. So we build a vocabulary, which is simply a list of every unique token in our text, with its own ID.

We also add two special tokens:

- an unknown token for any word the tokenizer has never seen before
- an end of text token to mark where one piece of text finishes and another begins

The size of this vocabulary tells us how many different tokens our tokenizer knows about.

In [8]:
all_words= sorted(list(set(preprocessed)))
all_words.extend(["<|endoftext|>", "<|unk|>"])
vocab_size= len(all_words)
print("unique vocab size:", vocab_size)

vocab = {word: integer for integer, word in enumerate(all_words)}

unique vocab size: 1161


## Putting it together: a simple tokenizer

Now we wrap everything into one small class. A tokenizer really only has two jobs:

- encode: take text and turn it into a list of IDs
- decode: take a list of IDs and turn it back into readable text

If it runs into a word that is not in its vocabulary, it falls back to the unknown token instead of crashing. The quick test at the bottom shows it turning a few IDs back into words.

In [9]:
class SimpleTokenizerv1:
    def __init__(self,vocab):
        self.vocab=vocab 
        self.inverse_vocab={v:k for k,v in vocab.items()}
    def encode(self, text):
        import re 
        preprocessed= re.split(r'([,.?!"()\']|--|\s)', text)
        preprocessed= [item.strip() for item in preprocessed if item.strip()]
        ids= [self.vocab.get(s, self.vocab["<|unk|>"]) for s in preprocessed]
        return ids
    def decode(self,ids):
        text= " ".join([self.inverse_vocab[i] for i in ids])

        import re 
        text= re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text 
tokenizer= SimpleTokenizerv1(vocab)
new_vocab= tokenizer.decode([1160, 1160, 155, 1160])
print(new_vocab)


<|unk|> <|unk|> am <|unk|>


## A smarter tokenizer: Byte Pair Encoding

Our simple tokenizer works, but it has one big weakness. The moment it meets a word that is not in its vocabulary, it gives up and returns the unknown token. Real language models cannot afford to do that.

This is where Byte Pair Encoding, usually shortened to BPE, comes in. Instead of memorising whole words, BPE breaks rare or unseen words into smaller pieces that it does know. So even a brand new word can still be represented. This is the same approach the GPT models use.

Rather than build BPE from scratch, we use a fast library called tiktoken, which gives us the exact tokenizer that GPT-2 uses. The next cell installs it.

In [10]:
%pip install --upgrade tiktoken

ERROR: Could not find a version that satisfies the requirement upgrade (from versions: none)
ERROR: No matching distribution found for upgrade
Note: you may need to restart the kernel to use updated packages.


Let us see it in action. We take a sentence, encode it into token IDs, and then decode those IDs straight back into text. Notice that even the odd made up word in there is handled fine without any unknown token, because BPE quietly breaks it into smaller pieces it already knows.

In [ ]:
import tiktoken
tokenizer= tiktoken.get_encoding("gpt2")
text = "hello world, yash is-learning-fast"
integers= tokenizer.encode(text)
print("encode IDS", integers)
print("decoded text", tokenizer.decode(integers))

ERROR: Could not find a version that satisfies the requirement upgrade (from versions: none)
ERROR: No matching distribution found for upgrade
Note: you may need to restart the kernel to use updated packages.
encode IDS [31373, 995, 11, 331, 1077, 318, 12, 40684, 12, 7217]
decoded text hello world, yash is-learning-fast


## Teaching the model to predict the next word

This is the core idea behind how a language model learns. We show it a few words and ask it to guess the word that comes next.

To set that up we take our list of tokens and make two versions of it, shifted by one position. The input holds the current tokens, and the target holds the token that follows each one. So if the input is "I", the target is "H", then "I H" points to "AD", and so on. The whole training process is really just the model getting better at this one simple guessing game, repeated millions of times.

In [12]:
with open("../data/the-verdict.txt" , "r", encoding="utf-8") as f:
    raw_text=f.read()

import tiktoken 
tokenizer= tiktoken.get_encoding("gpt2")
enc_text= tokenizer.encode(raw_text)
x= enc_text[:4]
y = enc_text[1:5]
print("x input IDs", x)
print("y output IDs", y)
for i in range (1, len(x)+1):
    context=x[:i]
    desired=y[i-1]
    print(tokenizer.decode(context), "----->", tokenizer.decode([desired]))

""" x diya 40 to y aya 367 samjha to hum llm ko first word se hi sikha rhe kaise karna hai ki 40 diya to 367 karna hai samjha fir
367 diya to 2885 an so on then 1464 diya to it became 1807"""

x input IDs [40, 367, 2885, 1464]
y output IDs [367, 2885, 1464, 1807]
I ----->  H
I H -----> AD
I HAD ----->  always
I HAD always ----->  thought


' x diya 40 to y aya 367 samjha to hum llm ko first word se hi sikha rhe kaise karna hai ki 40 diya to 367 karna hai samjha fir\n367 diya to 2885 an so on then 1464 diya to it became 1807'

## Creating a data loader for GPT

Training on one example at a time would be very slow. In practice we feed the model many sequences at once, in batches.

To keep this clean we will use two helpers from PyTorch. A Dataset to hold all of our input and target pairs, and a DataLoader to group them into batches for us automatically.

In [13]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


## The Dataset: sliding a window over the text

This class walks through the list of token IDs using a sliding window. At every step it takes a chunk of tokens as the input, and the same chunk shifted forward by one as the target.

Two settings control how this works:

- max_length is how many tokens go into each chunk
- stride is how far the window jumps each time

The end result is a tidy collection of input and target pairs that is ready to be batched.

In [14]:
import torch 
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids= []
        token_ids= tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk= token_ids[i:i + max_length]
            target_chunk= token_ids[i+1:i + max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self,idx):
        return self.input_ids[idx], self.target_ids[idx]

The function below is just a small convenience wrapper. You give it some text along with a few settings, and it builds the Dataset and wraps it inside a DataLoader for you. That way you get ready made batches without repeating all this setup every time.

In [15]:
import tiktoken
from torch.utils.data import DataLoader

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)
    return dataloader

## Looking at a real batch

Before handing anything to a model it is a good habit to check that the data looks the way we expect.

So here we pull one batch out of the data loader and print the shapes of the inputs and targets. Then we print the first input sequence and the first target sequence, so we can see those shifted by one pairs with our own eyes.

In [16]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text= f.read()
dataloader= create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter= iter(dataloader)
inputs,targets= next(data_iter)

print("Input tensors shape", inputs.shape)
print("targets tensor shape", targets.shape)
print("\nfirst input sequence in batch:\n", inputs[0])
print("\nfirst target sequence in batch:\n", targets[0])


Input tensors shape torch.Size([8, 4])
targets tensor shape torch.Size([8, 4])

first input sequence in batch:
 tensor([  40,  367, 2885, 1464])

first target sequence in batch:
 tensor([ 367, 2885, 1464, 1807])


## From IDs to vectors: the embedding layer

Token IDs are just labels. The number 40 is not bigger or more important than the number 5, they are only names for tokens. So we cannot feed raw IDs into a network and expect it to understand meaning from them.

Instead we use an embedding layer. The easiest way to picture it is a big lookup table: every token ID gets mapped to a list of numbers, called a vector, that the model can adjust while it learns.

Here each token becomes a vector of 256 numbers. So our batch of token IDs turns into a batch of vectors, and that is what the model actually works with.

In [18]:
import torch
import tiktoken

# inputs come from the tiktoken GPT-2 dataloader, so use GPT-2 vocab size
vocab_size = tiktoken.get_encoding("gpt2").n_vocab  # 50257
output_dim = 256

embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
embedded_output = embedding_layer(inputs)
print("original token ID shape:", inputs.shape)
print("Embedded vector shape:", embedded_output.shape)

original token ID shape: torch.Size([8, 4])
Embedded vector shape: torch.Size([8, 4, 256])


## Wrapping up

That is the full journey from raw text to something a model can actually learn from. Quick recap of what we covered:

- loaded a story as plain text
- split it into tokens and built a vocabulary
- wrote a simple tokenizer, then switched to BPE using tiktoken
- created input and target pairs for next word prediction
- batched everything with a Dataset and a DataLoader
- turned token IDs into embedding vectors

Next up is the interesting part: attention. That is the idea that lets the model decide which words matter most when it makes a prediction.